# 12 — Proyek Integratif Pengolahan Citra dan Machine Learning
### Dari pembentukan citra, koreksi radiometrik, segmentasi, dan fitur menuju keputusan yang dapat diaudit

**Tema utama:** inspeksi kualitas tablet dengan citra sensor sintetis dan pembagian data berdasarkan sesi akuisisi.

Notebook ini merupakan kelanjutan terintegrasi dari representasi citra (01), histogram (02), filtering (03), deteksi tepi (04), segmentasi (05), Fourier (06), warna (07), ekstraksi fitur (08), klasifikasi (09), deep learning (10), dan generalisasi (11). Keberhasilan tidak dinilai hanya dari nilai F1: sebuah pipeline juga perlu menghasilkan pengukuran yang masuk akal, ketidakpastian yang dilaporkan dengan benar, serta batas klaim yang jujur.

> **Status data:** semua citra dalam studi kasus dibuat oleh simulator. Satuan milimeter menggunakan *pixel pitch* asumsi; data ini bukan rekaman kamera atau inspeksi farmasi tervalidasi. Ground truth berasal dari proses pembangkitan data dan tidak boleh dipakai saat ekstraksi fitur atau pelatihan.

## Peta capaian dan diagnosis awal

Setelah mempelajari notebook ini, pembaca diharapkan dapat:

1. Merumuskan model maju pencitraan yang menghubungkan reflektansi, iluminasi, PSF, gain, offset, dan noise.
2. Menjelaskan mengapa pengukuran dark–flat dilakukan sebelum segmentasi dan ekstraksi fitur.
3. Mengukur kualitas segmentasi pada tingkat piksel dan konsekuensinya terhadap estimasi luas.
4. Membuat pipeline fitur dan klasifikasi tanpa kebocoran data antarsesi.
5. Memilih hiperparameter melalui data validasi lalu mengunci protokol pengujian.
6. Mengevaluasi akurasi, kalibrasi, domain shift, serta interval kepercayaan berbasis unit eksperimen.
7. Menyusun laporan ilmiah yang membedakan bukti numerik dari klaim generalisasi.

**Pertanyaan diagnostik:** Apakah citra yang lebih terang pasti memiliki informasi lebih banyak? Apakah Dice tinggi membuktikan luas objek tepat? Apakah F1 tinggi pada data test menjamin performa di kamera berbeda? Apa yang berubah jika lima gambar berasal dari satu tablet yang sama?

**Petunjuk:** jalankan dari atas ke bawah (*Restart & Run All*). Seluruh data dibangkitkan lokal; GPU maupun API berbayar tidak diperlukan. Grafik wajib ditafsirkan secara ilmiah.

## 1. Kontrak lingkungan, notasi, dan reproduksibilitas

Kontrak numerik: citra adalah array `float32` dalam skala intensitas ternormalisasi $[0,1]$; urutan koordinat adalah `(baris, kolom)` atau $(y,x)$; operasi geometri memakai piksel persegi dengan asumsi *pixel pitch* $p=0{,}045\,\mathrm{mm/pixel}$. Simulator memakai benih acak tetap dan seluruh variabel yang diperlukan didefinisikan di notebook ini.

Pustaka: NumPy, SciPy, scikit-image, scikit-learn, Matplotlib, dan pandas. Versi pustaka dapat memengaruhi pembulatan dan metrik marginal, sehingga angkanya perlu dicatat.

In [ ]:
import sys, platform, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy, sklearn, skimage
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr
from skimage.draw import disk, line
from skimage.morphology import closing, disk as disk_se
from skimage.measure import label, regionprops
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, balanced_accuracy_score, brier_score_loss,
                             precision_score, recall_score)
from sklearn.model_selection import ParameterGrid
SEED = 12026
PITCH_MM = 0.045
RNG = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':105,'axes.grid':False})
print({'python':sys.version.split()[0], 'numpy':np.__version__,
       'scipy':scipy.__version__, 'skimage':skimage.__version__,
       'scikit_learn':sklearn.__version__, 'matplotlib':matplotlib.__version__})

## 2. Dari permukaan tablet hingga angka digital: model maju

Permukaan memiliki reflektansi efektif $R(x,y)$. Sensor tidak melihat $R$ secara langsung: foton yang tiba bergantung pada medan iluminasi $L(x,y)$, *point spread function* $h$, penguatan $g$, dan offset gelap $d$. Model pengamatan tereduksi dapat ditulis

$$
Y(x,y)=\operatorname{clip}\!\left\{g\,\big[h*(L R)\big](x,y)+d+\eta(x,y)\right\}.
$$

Tanda $*$ menunjukkan konvolusi dua dimensi. Model ini adalah pendekatan fenomenologis pada respons intensitas linear, **bukan** model spektral lengkap kamera. Jika respons sensor nonlinier atau saturasi terjadi, invers yang mengasumsikan linearitas akan bias.

Dalam citra terkalibrasi, *background* tidak menghilang: kondisi pencahayaan, blur, variasi reflektansi, dan noise masih memengaruhi pemilahan objek. Batas fisik ini kelak menentukan apakah fitur geometri atau tekstur lebih stabil.

### 2.1 Noise, saturasi, dan identifiabilitas

Untuk intensitas foton yang dinyatakan sebagai ekspektasi elektron $\mu$, model dasar shot noise adalah $N\sim\operatorname{Poisson}(\mu)$. Jika dibaca dalam satuan intensitas ternormalisasi, pembacaan dapat diperkirakan sebagai

$$
Y = \frac{N}{n_\mathrm{full}} + \epsilon_r,\qquad
\operatorname{Var}(Y\mid\mu) = \frac{\mu}{n_\mathrm{full}^2}+\sigma_r^2.
$$

Asumsi tersebut menjelaskan noise yang bergantung sinyal, tetapi dalam simulator kita menggunakan pendekatan Poisson pada intensitas terukur untuk menjaga biaya komputasi. Variansi sumber yang berbeda tidak dapat disimpulkan unik hanya dari satu citra. Ketika piksel tersaturasi, informasi amplitudo yang hilang tidak dapat dipulihkan lewat normalisasi.

In [ ]:
# Ilustrasi mandiri: tren variansi foton dan noise baca.
photon_rng=np.random.default_rng(2026)
levels=np.linspace(.04,.9,10)
variances=[]
for level in levels:
    e=photon_rng.poisson(level*2500,size=1500)/2500
    s=e+photon_rng.normal(0,0.007,size=e.shape)
    variances.append(float(np.var(s,ddof=1)))
fig,ax=plt.subplots(figsize=(7,3.7))
ax.scatter(levels,variances,label='Simulasi variansi')
ax.plot(levels,levels/2500+0.007**2,'--',label='Model Poisson + baca')
ax.set(xlabel='Intensitas ekspektasi (ternormalisasi)',ylabel='Variansi intensitas',title='Noise tidak selalu memiliki variansi konstan')
ax.legend();plt.show()

## 3. Rancangan eksperimen: spesimen, sesi akuisisi, dan tujuan

**Unit analisis:** satu citra tablet. **Unit independen untuk validasi:** satu sesi/batch akuisisi; setiap batch mempunyai dark/flat, gain, dan pola iluminasi yang sama sehingga delapan citra dari satu batch berkorelasi. Jika citra dibagi secara acak tanpa memperhatikan batch, contoh dari sensor dan iluminasi yang sama dapat masuk train dan test sehingga performa tampak terlalu optimistis.

Dataset utama menggunakan 40 batch (24 train, 8 validation, 8 test), masing-masing memuat 8 citra dengan komposisi kelas seimbang. Satu set *stress* berisi 8 batch lain, tanpa batch yang sama, dengan perubahan gain, noise, blur, dan iluminasi. Kelas 1 berarti **memiliki cacat visual sintetis** (retakan atau rongga); kelas 0 berarti tanpa cacat. Perbedaan antarbatch dirancang independen dari label agar tidak muncul jalan pintas melalui identitas batch.

Urutan keputusan:

1. Definisikan model sensor dan buat data.
2. Pisahkan batch; jangan mengintip test untuk tuning.
3. Tentukan prapemrosesan menggunakan train/validation.
4. Evaluasi segmentasi serta keluaran pengukuran.
5. Ekstrak fitur hanya dari piksel dan referensi dark/flat.
6. Bandingkan classifier; pilih dari validation.
7. Laporkan test normal dan stress sekali dengan konfigurasi terpilih.
8. Evaluasi interval kepercayaan, kesalahan, dan batas generalisasi.

### 3.1 Generator citra: mask kebenaran dan sinyal sensor

Tablet dibuat sebagai cakram dengan tepi tidak sepenuhnya ideal, rongga atau retakan sintetis, tekstur berfrekuensi rendah, dan posisi yang sedikit bergeser. *Ground truth* geometri hanya digunakan untuk menguji segmentasi, **tidak** diberikan kepada model sebagai fitur. Simulasi dikendalikan seed sehingga dapat direproduksi.

Bentuk gambar bukan simulasi optik kuantitatif yang lengkap: ukuran PSF, spektrum iluminan, persamaan respons kamera, dan pola retakan harus dikalibrasi jika ingin berpindah ke instrumen nyata.

In [ ]:
H=W=64
YY, XX=np.mgrid[:H,:W]

def make_batch(batch_id, split, seed=SEED):
    # Satu RNG independen untuk setiap batch: perubahan urutan eksekusi tidak mengubah isinya.
    rng=np.random.default_rng(seed+batch_id*997)
    stress=(split=='stress')
    yy=(YY-(H-1)/2)/H
    xx=(XX-(W-1)/2)/W
    gain=rng.uniform(.82,1.10) if not stress else rng.uniform(.66,1.25)
    dark_level=rng.uniform(.025,.052) if not stress else rng.uniform(.02,.07)
    gradients=rng.uniform(-.22,.22,2) if not stress else rng.uniform(-.50,.50,2)
    illum=np.clip(1+gradients[0]*xx+gradients[1]*yy, .4,1.6)
    illum*=1-(rng.uniform(.05,.14) if not stress else rng.uniform(.12,.30))*(xx**2+yy**2)*4
    illum=np.clip(illum,.48,1.45)
    blur=float(rng.uniform(.55,1.05) if not stress else rng.uniform(1.0,1.9))
    electrons=4200 if not stress else 1800
    read_noise=.006 if not stress else .011
    offset=dark_level+.002*xx
    # Dark dan flat sebagai kalibrasi sesi, bukan gambar ground truth.
    def noise_signal(expected):
        expected=np.clip(expected,0,1)
        return np.clip(rng.poisson(expected*electrons)/electrons+rng.normal(0,read_noise,(H,W)),0,1).astype('float32')
    dark=noise_signal(offset)
    flat=noise_signal(gain*illum*.70+offset)
    images=[]; y=[]; true_masks=[]; refls=[]; meta=[]
    labels=np.array([0]*4+[1]*4)
    rng.shuffle(labels)
    for j,cls in enumerate(labels):
        cx=31.5+rng.uniform(-2.5,2.5)
        cy=31.5+rng.uniform(-2.5,2.5)
        radius=rng.uniform(16.0,19.0)
        r=np.sqrt(((XX-cx)/rng.uniform(.95,1.06))**2+((YY-cy)/rng.uniform(.95,1.06))**2)
        tablet=(r<radius)
        if cls==1:
            if j%2==0:
                cr,cc=disk((cy+rng.uniform(-6,6),cx+rng.uniform(-6,6)),rng.uniform(2.4,4.2),shape=(H,W))
                flaw=np.zeros((H,W),bool); flaw[cr,cc]=True
            else:
                flaw=np.zeros((H,W),bool)
                y0=int(cy+rng.uniform(-6,6)); x0=int(cx+rng.uniform(-6,6))
                rr,cc=line(y0,max(0,x0-9),min(H-1,y0+rng.integers(-4,5)),min(W-1,x0+10))
                flaw[np.clip(rr,0,H-1),np.clip(cc,0,W-1)]=True
                flaw=closing(flaw,disk_se(1))
            flaw &= tablet
        else:
            flaw=np.zeros((H,W),bool)
        texture=gaussian_filter(rng.normal(0,1,(H,W)),2)
        texture=texture/(np.std(texture)+1e-9)*rng.uniform(.012,.028)
        true_refl=np.full((H,W),.19,dtype=float)
        true_refl[tablet]=.70+texture[tablet]
        true_refl[flaw]-=rng.uniform(.10,.34)
        true_refl=np.clip(true_refl,.02,.95)
        expected=gain*gaussian_filter(illum*true_refl,blur)+offset
        observed=noise_signal(expected)
        images.append(observed); y.append(int(cls)); true_masks.append(tablet)
        refls.append(true_refl.astype('float32'))
        meta.append((batch_id,j,split,float(gain),float(blur)))
    return (np.stack(images),np.asarray(y),np.stack(true_masks),
            np.stack(refls),np.stack([dark]*8),np.stack([flat]*8),meta)

b=make_batch(0,'train')
assert b[0].shape==(8,64,64) and np.sum(b[1])==4
print('Satu batch:',b[0].shape,'citra; label:',np.bincount(b[1]))

### 3.2 Audit visual: variasi fisik atau petunjuk label yang tidak sengaja?

Visualisasi dilakukan untuk mengenali jenis degradasi dan memeriksa apakah label tidak tersandi secara langsung melalui latar belakang, ukuran, atau batch. **Inspeksi visual saja tidak cukup membuktikan tidak ada shortcut**: kelak diperlukan uji pada data nyata dan analisis confounder.

In [ ]:
fig,axs=plt.subplots(2,4,figsize=(12,6))
for ax,im,cls in zip(axs.flat,b[0],b[1]):
    ax.imshow(im,cmap='gray',vmin=0,vmax=1)
    ax.set_title('Cacat' if cls else 'Normal');ax.axis('off')
fig.suptitle('Variasi citra dalam satu batch sintetis');plt.tight_layout();plt.show()

## 4. Metadata, pemisahan berdasarkan batch, dan integritas data

Pembagian yang sah menghindari irisan batch: $B_{\mathrm{train}}\cap B_{\mathrm{validation}}=\varnothing$, demikian pula dengan himpunan lain. Karena ada delapan citra dalam satu sesi, ukuran sampel efektif untuk memperkirakan generalisasi antarsesi lebih dekat ke jumlah batch daripada jumlah gambar.

Kita menggunakan pembagian batch tetap dan menyimpan identitasnya di manifest. Identitas batch tidak digunakan sebagai prediktor. Keberimbangan label di setiap batch adalah pilihan desain simulator; prevalensi cacat di lapangan mungkin sangat berbeda.

In [ ]:
def join_batches(batch_ids,split):
    parts=[make_batch(i,split) for i in batch_ids]
    return {'images':np.concatenate([p[0] for p in parts]),
            'labels':np.concatenate([p[1] for p in parts]),
            'truth_masks':np.concatenate([p[2] for p in parts]),
            'truth_refl':np.concatenate([p[3] for p in parts]),
            'dark':np.concatenate([p[4] for p in parts]),
            'flat':np.concatenate([p[5] for p in parts]),
            'meta':sum([p[6] for p in parts],[])}
BATCHES={'train':list(range(0,24)), 'validation':list(range(24,32)),
         'test':list(range(32,40)), 'stress':list(range(40,48))}
datasets={s:join_batches(ids,s) for s,ids in BATCHES.items()}
for a in BATCHES:
    for bname in BATCHES:
        if a!=bname:
            assert not (set(BATCHES[a]) & set(BATCHES[bname]))
for name,data in datasets.items():
    print(f'{name:10s} : {len(data["labels"]):3d} citra; {len(BATCHES[name])} batch; distribusi {np.bincount(data["labels"]).tolist()}')

## 5. Koreksi dark–flat field: dari asumsi hingga persamaan

Jika *dark frame* $D\approx d$ dan *flat frame* merekam permukaan homogen dengan reflektansi diketahui $R_0$, maka

$$
Y-D\approx gLR,\qquad F-D\approx gLR_0.
$$

Estimasi reflektansi (dalam batas asumsi model) adalah

$$
\widehat R(x,y)=R_0\frac{Y(x,y)-D(x,y)}{F(x,y)-D(x,y)+\varepsilon}.
$$

$\varepsilon$ mencegah pembagian oleh nol secara numerik; ia **tidak** menyelesaikan masalah piksel mati atau flat yang tidak akurat. Gaussian blur, noise, saturasi, perubahan lampu antarreferensi, dan ketidaklinearan sensor menyebabkan estimasi tetap tidak sempurna. Kalibrasi warna spektral dan penentuan reflektansi absolut memerlukan protokol lebih ketat daripada contoh ini.

Perhatikan bahwa *dark* dan *flat* merupakan pengukuran pendukung dari batch yang sama. Menggunakan keduanya pada sesi test sah jika protokol akuisisi memang menyediakan referensi kalibrasi pada saat deployment. Menerapkan referensi dari masa depan atau memilih parameternya dari label test tidak sah.

In [ ]:
def dark_flat_correct(images,dark,flat,reference=.70):
    denom=np.maximum(flat-dark,.05)
    return np.clip(reference*(images-dark)/denom,0,1).astype('float32')

for name in datasets:
    d=datasets[name]
    d['corrected']=dark_flat_correct(d['images'],d['dark'],d['flat'])
fig,axes=plt.subplots(2,4,figsize=(13,6))
for j,s in enumerate(['train','validation','test','stress']):
    d=datasets[s]; i=0
    axes[0,j].imshow(d['images'][i],cmap='gray',vmin=0,vmax=1)
    axes[1,j].imshow(d['corrected'][i],cmap='gray',vmin=0,vmax=1)
    axes[0,j].set_title(f'{s}: sensor mentah');axes[1,j].set_title(f'{s}: dark–flat')
    for ax in [axes[0,j],axes[1,j]]: ax.axis('off')
plt.tight_layout();plt.show()

### 5.1 Audit radiometri: membandingkan estimasi dengan reflektansi referensi sintetis

Karena simulator menyimpan $R$ yang diketahui, kita dapat menghitung galat piksel sebagai pemeriksaan numerik. Evaluasi ini **bukan** metrik kinerja lapangan: tidak mungkin mengetahui reflektansi absolut dari satu citra nyata tanpa referensi. Di sini digunakan *mean absolute error* sebagai

$$
\operatorname{MAE}_R=\frac{1}{N}\sum_{i=1}^{N}\left|\widehat R_i-R_i\right|.
$$

MAE pada permukaan tablet bisa lebih relevan untuk pemodelan cacat daripada MAE global, karena latar belakang sering mendominasi jumlah piksel.

In [ ]:
rad_rows=[]
for s,d in datasets.items():
    roi=d['truth_masks']
    for method,image in [('mentah',d['images']),('dark–flat',d['corrected'])]:
        err=np.abs(image-d['truth_refl'])
        rad_rows.append({'split':s,'metode':method,'MAE global':float(err.mean()),
                         'MAE tablet':float(err[roi].mean())})
rad_df=pd.DataFrame(rad_rows)
print(rad_df.round(4).to_string(index=False))

## 6. Segmentasi sebagai masalah keputusan dan pengukuran

Segmentasi menentukan himpunan piksel $\widehat M=\{(x,y): \widehat R(x,y)>t\}$. Ambang $t$ tidak mempunyai makna universal: ia bergantung pada objek, latar, kalibrasi, dan noise. *Morphological closing* menutup lubang kecil sehingga cacat internal tidak diperlakukan sebagai penghapusan seluruh objek. Penyaringan komponen menghilangkan artefak kecil yang terpisah.

Untuk mask biner $M$ dan prediksi $\widehat M$, ukuran tumpang tindih adalah

$$
\operatorname{Dice}=\frac{2|M\cap\widehat M|}{|M|+|\widehat M|},\qquad
\operatorname{IoU}=\frac{|M\cap\widehat M|}{|M\cup\widehat M|}.
$$

Keduanya tidak mengukur kesalahan dalam satuan mm²; dua segmentasi dengan Dice serupa dapat memiliki bias luas yang berbeda. Oleh karena itu, ketiga ukuran harus dilaporkan jika tujuan akhirnya adalah metrologi.

In [ ]:
from scipy.ndimage import binary_fill_holes

def segment_tablet(im,threshold=.39):
    mask=np.asarray(im>threshold)
    mask=closing(mask,disk_se(2))
    mask=binary_fill_holes(mask)
    components=label(mask,connectivity=2)
    if components.max()==0: return np.zeros_like(mask,dtype=bool)
    sizes=np.bincount(components.ravel()); sizes[0]=0
    return components==np.argmax(sizes)

def dice(a,b):
    denom=np.sum(a)+np.sum(b)
    return (2*np.sum(a&b)/denom) if denom>0 else 1.

def iou(a,b):
    return np.sum(a&b)/max(np.sum(a|b),1)

# Ambang dan opsi prapemrosesan dipilih HANYA dari validation.
# Train bukan test dan label 'truth_masks' hanya dipakai pada tahap pemilihan ini.
results=[]
for method in ('images','corrected'):
    for t in np.linspace(.31,.48,18):
        val=datasets['validation']
        ds=[dice(segment_tablet(x,t),m) for x,m in zip(val[method],val['truth_masks'])]
        results.append({'metode':method,'ambang':float(t),'Dice validasi':float(np.mean(ds))})
seg_val=pd.DataFrame(results)
selected_seg={}
for method in ('images','corrected'):
    row=seg_val[seg_val.metode==method].sort_values(['Dice validasi','ambang'],ascending=[False,True]).iloc[0]
    selected_seg[method]=float(row['ambang'])
print('Ambang dari validasi:',selected_seg)
print(seg_val.groupby('metode')['Dice validasi'].max().to_string())

### 6.1 Menguji mask dan menilai kegagalan yang tampak

Perbedaan antara dua metode sering terlihat pada kontur batas, bukan pada pusat objek. Pengecekan label *ground truth* saat analisis kesalahan diperbolehkan setelah evaluasi; tetapi mengubah parameter berdasarkan kesalahan data test berarti melakukan tuning terselubung.

In [ ]:
d=datasets['test']; idx=3
f,ax=plt.subplots(1,4,figsize=(13,3.3))
ax[0].imshow(d['images'][idx],cmap='gray',vmin=0,vmax=1);ax[0].set_title('Citra mentah')
ax[1].imshow(d['truth_masks'][idx],cmap='gray');ax[1].set_title('Mask kebenaran')
for a,k in zip(ax[2:],['images','corrected']):
    pred=segment_tablet(d[k][idx],selected_seg[k]);a.imshow(pred,cmap='gray')
    a.set_title(f'{k}, Dice={dice(pred,d["truth_masks"][idx]):.3f}')
for a in ax:a.axis('off')
plt.tight_layout();plt.show()

## 7. Metrologi: luas, bias, varians, dan batas kalibrasi

Dengan *pixel pitch* $p_x$ dan $p_y$, luas satu piksel adalah $p_xp_y$. Untuk mask berisi $N_M$ piksel,

$$
\widehat A=N_{\widehat M}\,p_xp_y.
$$

Jika kedua pitch sama $p$, maka $\widehat A=N_{\widehat M}p^2$. Bias akibat pergeseran batas kira-kira bergantung pada panjang perimeter objek; kesalahan satu piksel di sekeliling tepi dapat lebih besar daripada noise rerata interior. Asumsi *pixel pitch* diketahui tepat adalah penyederhanaan. Ketika $p_x$ dan $p_y$ mempunyai ketidakpastian, propagasi orde pertama untuk $A=Np_xp_y$ mencakup suku turunan terhadap ketiganya dan kovarians apabila tidak independen.

Kita melaporkan MAE luas dan bias rata-rata dalam mm². Nilai-nilai ini mengacu ke skala sintetis, **bukan sertifikat kalibrasi metrologi**.

In [ ]:
seg_metrics=[]
for s,d in datasets.items():
    for k in ('images','corrected'):
        preds=np.array([segment_tablet(im,selected_seg[k]) for im in d[k]])
        d['masks_'+k]=preds
        truth_area=d['truth_masks'].sum(axis=(1,2))*PITCH_MM**2
        pred_area=preds.sum(axis=(1,2))*PITCH_MM**2
        d['area_'+k]=pred_area
        ds=np.array([dice(a,b) for a,b in zip(preds,d['truth_masks'])])
        errors=pred_area-truth_area
        seg_metrics.append({'split':s,'metode':k,'Dice':float(ds.mean()),
            'MAE luas (mm²)':float(np.abs(errors).mean()),'Bias luas (mm²)':float(errors.mean())})
seg_summary=pd.DataFrame(seg_metrics)
print(seg_summary.round(4).to_string(index=False))

## 8. Ekstraksi fitur dengan asumsi domain yang eksplisit

Model klasik tidak menerima label kebenaran: ia hanya menggunakan hasil dari pipeline yang memang akan tersedia pada penggunaan nyata. Contoh fitur:

- luas dan keliling terukur dari mask prediksi;
- intensitas rerata dan simpangan baku di wilayah tablet;
- selisih intensitas terhadap fraksi gelap (indikator rongga/retak);
- jumlah piksel gelap interior;
- statistik gradien dan tekstur di interior objek.

Fitur yang memakai **mask kebenaran** saat melatih classifier akan menciptakan representasi tidak realistis jika kebenaran tersebut tidak tersedia pada pemakaian nyata. Demikian pula identitas batch, jenis cacat simulator, dan reflektansi tersembunyi harus dilarang sebagai input. Fitur dihitung konsisten untuk train, validation, test, dan stress.

In [ ]:
from scipy.ndimage import binary_erosion, sobel
FEATURE_NAMES=['luas_px','keliling_px','mean_R','std_R','p10_R','fraksi_gelap',
               'std_gradien','kontras_dalam','compactness']

def features_from_image(im,mask):
    mask=mask.astype(bool)
    interior=binary_erosion(mask,iterations=3)
    if interior.sum()<100:interior=mask.copy()
    values=im[interior]
    if len(values)==0:return np.zeros(len(FEATURE_NAMES),dtype=float)
    p10=np.percentile(values,10)
    gy=sobel(im,axis=0);gx=sobel(im,axis=1)
    grad=np.hypot(gx,gy)
    props=regionprops(mask.astype('uint8'))
    perimeter=float(props[0].perimeter) if props else 0.
    area=float(mask.sum());compactness=4*np.pi*area/(perimeter**2+1e-8)
    return np.array([area,perimeter,float(np.mean(values)),float(np.std(values)),float(p10),
        float(np.mean(values<.44)),float(np.std(grad[interior])),
        float(np.percentile(values,90)-p10),compactness])

feature_banks={}
for kind in ('images','corrected'):
    feature_banks[kind]={}
    for split,d in datasets.items():
        X=np.stack([features_from_image(im,m) for im,m in zip(d[kind],d['masks_'+kind])])
        assert np.isfinite(X).all()
        feature_banks[kind][split]=X
print('Matriks fitur train:',feature_banks['corrected']['train'].shape)
print(pd.DataFrame(feature_banks['corrected']['train'],columns=FEATURE_NAMES).describe().loc[['mean','std']].round(3).to_string())

### 8.1 Audit fitur: korelasi dan masalah redundansi

Fitur yang sangat berkorelasi dapat membuat interpretasi koefisien regresi logistik tidak stabil. Standarisasi fitur harus menggunakan statistik data **train saja**. Di sini korelasi dilihat sebagai diagnosis, bukan alasan otomatis menghapus fitur dari test. Keputusan reduksi fitur harus ditetapkan dengan validasi sebelum melihat hasil final.

In [ ]:
corr=pd.DataFrame(feature_banks['corrected']['train'],columns=FEATURE_NAMES).corr()
fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(corr,cmap='coolwarm',vmin=-1,vmax=1)
ax.set_xticks(range(len(FEATURE_NAMES)),FEATURE_NAMES,rotation=70,ha='right')
ax.set_yticks(range(len(FEATURE_NAMES)),FEATURE_NAMES)
fig.colorbar(im,ax=ax,label='Pearson r');ax.set_title('Korelasi fitur pada data train')
plt.tight_layout();plt.show()

## 9. Dari fitur ke probabilitas keputusan

Model regresi logistik biner mempunyai bentuk

$$
P_\theta(Y=1\mid\mathbf x)=\sigma(\mathbf w^\top\mathbf x+b),\qquad
\sigma(z)=\frac{1}{1+\exp(-z)}.
$$

Parameter dapat diestimasi dengan meminimalkan *binary cross entropy* yang diregularisasi:

$$
\mathcal J(\theta)=-\frac{1}{N}\sum_{i=1}^{N}
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right]+
\frac{\lambda}{2}\|\mathbf w\|_2^2.
$$

Di scikit-learn, `C` mengatur kebalikan kekuatan regularisasi, dengan detail skala fungsi objektif mengikuti implementasi pustaka. Random forest menjadi pembanding nonlinier; *dummy classifier* mengukur nilai prediksi tanpa belajar fitur. Semua kandidat melihat train yang sama dan dinilai pada validation yang sama.

**Aturan keputusan:** gunakan macro-F1 validation untuk memilih konfigurasi. Hanya setelah model terpilih, test dan stress dievaluasi. Jangan memilih ulang berdasarkan test.

In [ ]:
Xtr=feature_banks['corrected']['train']; ytr=datasets['train']['labels']
Xval=feature_banks['corrected']['validation']; yval=datasets['validation']['labels']
candidates={'dummy': DummyClassifier(strategy='most_frequent'),
    **{f'logreg C={c}':make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=1200,random_state=SEED)) for c in [.01,.1,1.,10.]},
    **{f'forest leaf={leaf}':RandomForestClassifier(n_estimators=120,min_samples_leaf=leaf,random_state=SEED,n_jobs=1) for leaf in [2,5]}}
validation_results=[]
for name,model in candidates.items():
    t=time.perf_counter();model.fit(Xtr,ytr)
    pred=model.predict(Xval)
    validation_results.append({'model':name,'macro-F1':f1_score(yval,pred,average='macro'),
                               'balanced accuracy':balanced_accuracy_score(yval,pred),
                               'fit detik':time.perf_counter()-t})
val_scores=pd.DataFrame(validation_results)
val_scores['simplicity']=val_scores['model'].apply(lambda v: (0 if v.startswith('logreg') else 1 if v.startswith('forest') else 2))
val_scores=val_scores.sort_values(['macro-F1','simplicity','model'],ascending=[False,True,True]).reset_index(drop=True)
selected_name=str(val_scores.iloc[0]['model'])
selected_model=candidates[selected_name]
print(val_scores.round(4).to_string(index=False))
print('\nModel terkunci berdasarkan VALIDASI:',selected_name)

## 10. Pengujian final: confusion matrix, sensitivitas, dan kesalahan

Setelah model dikunci, kita mengevaluasi test menggunakan label yang belum pernah digunakan untuk pemilihan. Interpretasi harus memuat kesalahan kelas 0 dan kelas 1. Macro-F1 relevan karena memberi bobot yang sama kepada dua kelas. Pada aplikasi dengan prevalensi cacat sangat rendah, *precision* akan berubah sekalipun sensitivity tetap, sehingga hasil ini tidak langsung dipindahkan ke prevalensi lapangan.

In [ ]:
def eval_classifier(model,X,y,name):
    pred=model.predict(X)
    return {'split':name,'accuracy':accuracy_score(y,pred),
            'balanced_acc':balanced_accuracy_score(y,pred),
            'macro-F1':f1_score(y,pred,average='macro',zero_division=0),
            'precision_cacat':precision_score(y,pred,zero_division=0),
            'recall_cacat':recall_score(y,pred,zero_division=0)}
metrics=[]
for s in ['validation','test','stress']:
    metrics.append(eval_classifier(selected_model,feature_banks['corrected'][s],datasets[s]['labels'],s))
model_summary=pd.DataFrame(metrics)
print(model_summary.round(4).to_string(index=False))
ytest=datasets['test']['labels'];y_pred=selected_model.predict(feature_banks['corrected']['test'])
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(ytest,y_pred,display_labels=['Normal','Cacat'],cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Confusion matrix — TEST terkunci');plt.tight_layout();plt.show()

### 10.1 Inspeksi kegagalan bukan kesempatan tuning test

Satu prediksi salah dapat disebabkan oleh kesalahan prapemrosesan, segmentasi, representasi fitur, label, atau batas keputusan. Untuk satu kesalahan yang dipilih *post hoc*, gambarnya hanya ilustrasi diagnosis; bukan estimasi frekuensi jenis kegagalan di populasi. Jangan memperbaiki ambang dari gambar test lalu melaporkan skor yang sama sebagai evaluasi independen.

In [ ]:
wrong=np.where(y_pred!=ytest)[0]
if len(wrong):
    picks=wrong[:min(4,len(wrong))]
    fig,axes=plt.subplots(1,len(picks),figsize=(3.2*len(picks),3.2))
    for a,i in zip(np.atleast_1d(axes),picks):
        a.imshow(datasets['test']['corrected'][i],cmap='gray',vmin=0,vmax=1)
        a.set_title(f'Truth={ytest[i]}, pred={y_pred[i]}');a.axis('off')
    plt.tight_layout();plt.show()
else:
    print('Tidak ada salah klasifikasi pada split test ini; analisis kegagalan dibahas melalui stress dan segmentasi.')

## 11. Ablasi: bagian mana yang memberi kontribusi?

*Accuracy* saja tidak menunjukkan mengapa model berhasil. Uji ablasi membandingkan kelompok fitur dengan kondisi pelatihan, pembagian, dan evaluasi validation yang sama. Tidak ada kesimpulan kausal otomatis: menghapus satu fitur dapat mengubah efek fitur lainnya karena korelasi. Evaluasi ablasi di sini dilakukan di **validation**, bukan berkali-kali di test untuk memilih fitur terbaik.

In [ ]:
ablations={
    'semua':list(range(9)),
    'geometri':[0,1,8],
    'radiometri':[2,3,4,5,7],
    'tekstur-gradien':[3,6,7],
    'tanpa fraksi gelap':[0,1,2,3,4,6,7,8]}
abl=[]
for name,idxs in ablations.items():
    mod=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1200,random_state=SEED))
    mod.fit(Xtr[:,idxs],ytr)
    f1=f1_score(yval,mod.predict(Xval[:,idxs]),average='macro')
    abl.append({'fitur':name,'n_fitur':len(idxs),'F1-validasi':f1})
abl_df=pd.DataFrame(abl).sort_values('F1-validasi',ascending=False)
print(abl_df.round(4).to_string(index=False))
fig,ax=plt.subplots(figsize=(7,3.6))
ax.barh(abl_df['fitur'],abl_df['F1-validasi'])
ax.set(xlim=(0,1),xlabel='Macro-F1 pada validation',title='Ablasi representasi fitur')
plt.tight_layout();plt.show()

## 12. Uji domain shift: apakah perbaikan berlaku pada perangkat lain?

Domain *stress* memakai distribusi optik dan elektronik yang digeser, tetapi kelas dan prosedur pembangkitan tablet serupa. Kita menilai stabilitas tanpa mempelajari ulang hiperparameter dari stress. Jika kinerja turun, penyebabnya dapat berupa pergeseran citra, segmentasi, maupun fitur. Koreksi dark–flat membantu mengurangi satu sumber variasi, namun tidak membalikkan blur atau memulihkan piksel tersaturasi.

Keberhasilan pada domain sintetis yang didefinisikan penulis **bukan** bukti generalisasi lintas merek kamera, pabrik, atau kondisi produksi nyata.

In [ ]:
stress_pred=selected_model.predict(feature_banks['corrected']['stress'])
fig,axs=plt.subplots(1,2,figsize=(10,3.6))
for ax,s,pred in [(axs[0],'test',y_pred),(axs[1],'stress',stress_pred)]:
    ConfusionMatrixDisplay.from_predictions(datasets[s]['labels'],pred,
            display_labels=['Normal','Cacat'],cmap='Blues',colorbar=False,ax=ax)
    ax.set_title(f'Performa domain {s}')
plt.tight_layout();plt.show()
print('Gap macro-F1 test–stress:',round(model_summary.loc[model_summary.split=='test','macro-F1'].iloc[0]-
                                       model_summary.loc[model_summary.split=='stress','macro-F1'].iloc[0],4))

## 13. Ablasi pipeline: apakah koreksi radiometrik meningkatkan hasil akhir?

Kita bandingkan model linear dengan prosedur yang identik pada fitur mentah dan terkalibrasi, menggunakan batch yang persis sama. Nilai `C=1` dibakukan untuk eksperimen ablasi ini. Perbandingan bukan evaluasi otomatis terhadap CNN atau model lain. Perlu dibedakan tiga sasaran: kesalahan radiometri, kesalahan luas, dan kesalahan klasifikasi. Perbaikan satu metrik mungkin tidak mengubah metrik lainnya.

In [ ]:
compare=[]
for kind in ['images','corrected']:
    mod=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1200,random_state=SEED))
    mod.fit(feature_banks[kind]['train'],ytr)
    for split in ['validation','test','stress']:
        compare.append({'pipeline':kind,'split':split,
                       'macro-F1':f1_score(datasets[split]['labels'],mod.predict(feature_banks[kind][split]),average='macro')})
comparison=pd.DataFrame(compare)
print(comparison.pivot(index='pipeline',columns='split',values='macro-F1').round(4).to_string())
fig,ax=plt.subplots(figsize=(7.5,3.6))
x=np.arange(3);width=.36
for i,kind in enumerate(['images','corrected']):
    vals=[float(comparison[(comparison.pipeline==kind)&(comparison.split==s)]['macro-F1'].iloc[0]) for s in ['validation','test','stress']]
    ax.bar(x+(i-.5)*width,vals,width,label=kind)
ax.set(xticks=x,xticklabels=['Validation','Test','Stress'],ylim=(0,1),ylabel='Macro-F1',title='Ablasi koreksi radiometrik (C tetap)')
ax.legend();plt.tight_layout();plt.show()

## 14. Ketidakpastian kinerja: unit bootstrap adalah batch, bukan piksel

Jika citra dalam satu batch berbagi iluminasi dan gain, mengambil sampel bootstrap per gambar mengabaikan korelasi dan dapat menghasilkan interval yang terlalu sempit. Kita melakukan *cluster bootstrap* dengan mengambil **batch utuh** secara berulang. Pada setiap replikasi, seluruh citra dari batch terpilih diikutkan. Metrik dihitung ulang menggunakan prediksi model tetap; tidak ada retraining selama bootstrap.

Untuk statistik $T$, interval persentil ilustratif adalah

$$
\mathrm{CI}_{95\%}=[Q_{0.025}(T^*),\ Q_{0.975}(T^*)].
$$

Interval bootstrap dengan delapan batch test hanya menggambarkan ketidakpastian pengambilan sampel pada populasi sintetis yang diasumsikan sejenis; tidak mencakup bias generator, keliru label, atau perubahan perangkat dunia nyata.

In [ ]:
def bootstrap_batches(y,pred,batch_ids,n=1200,seed=124):
    rng=np.random.default_rng(seed)
    ids=np.unique(batch_ids)
    scores=[]
    for _ in range(n):
        sampled=rng.choice(ids,size=len(ids),replace=True)
        ii=np.concatenate([np.where(batch_ids==v)[0] for v in sampled])
        scores.append(f1_score(y[ii],pred[ii],average='macro',zero_division=0))
    return np.asarray(scores)

test_batches=np.array([t[0] for t in datasets['test']['meta']])
stress_batches=np.array([t[0] for t in datasets['stress']['meta']])
boot_test=bootstrap_batches(ytest,y_pred,test_batches)
boot_stress=bootstrap_batches(datasets['stress']['labels'],stress_pred,stress_batches,seed=125)
for s,v in [('test',boot_test),('stress',boot_stress)]:
    print(s, 'CI95% macro-F1:',np.quantile(v,[.025,.975]).round(4).tolist())
fig,ax=plt.subplots(figsize=(7,3.4))
ax.hist(boot_test,bins=17,alpha=.7,label='test');ax.hist(boot_stress,bins=17,alpha=.65,label='stress')
ax.set(xlabel='Macro-F1 bootstrap',ylabel='Jumlah resample',title='Ketidakpastian antar-batch (simulasi)')
ax.legend();plt.tight_layout();plt.show()

### 14.1 Interval berpasangan untuk beda performa

Perbandingan dua pipeline harus memasangkan prediksi pada batch yang sama. Dalam *paired cluster bootstrap*, indeks batch dipilih bersama untuk kedua model sehingga korelasi antarhasil tidak hilang. Interval yang melintasi nol menunjukkan bahwa data ini tidak cukup kuat untuk memastikan tanda perbedaan, tetapi ketidakpastian statistik tetap bukan satu-satunya ukuran manfaat rekayasa.

In [ ]:
model_raw=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1200,random_state=SEED))
model_flat=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1200,random_state=SEED))
model_raw.fit(feature_banks['images']['train'],ytr)
model_flat.fit(feature_banks['corrected']['train'],ytr)
pr=model_raw.predict(feature_banks['images']['test'])
pf=model_flat.predict(feature_banks['corrected']['test'])
rng=np.random.default_rng(991)
unique=np.unique(test_batches)
diffs=[]
for _ in range(1200):
    sample=rng.choice(unique,size=len(unique),replace=True)
    ii=np.concatenate([np.where(test_batches==k)[0] for k in sample])
    diffs.append(f1_score(ytest[ii],pf[ii],average='macro',zero_division=0)-f1_score(ytest[ii],pr[ii],average='macro',zero_division=0))
print('Δ F1(flat–raw) pada test =',round(f1_score(ytest,pf,average='macro')-f1_score(ytest,pr,average='macro'),4))
print('CI bootstrap berpasangan 95% =',np.quantile(diffs,[.025,.975]).round(4).tolist())

## 15. Kalibrasi probabilitas: keyakinan model bukan jaminan benar

Probabilitas keluaran regresi logistik adalah skor bersyarat pada model dan fiturnya. Nilai $p=0{,}9$ tidak otomatis berarti akurasi empiris 90%. Uji sederhana memakai *Brier score*, $\tfrac1N\sum_i(p_i-y_i)^2$, serta *reliability diagram* dengan bin probabilitas. Jumlah batch kecil dan kelas seimbang membatasi ketelitian penilaian; tidak boleh menyamakan *reliability diagram* ini dengan pengukuran ketidakpastian epistemik yang lengkap.

In [ ]:
if hasattr(selected_model,'predict_proba'):
    proba=selected_model.predict_proba(feature_banks['corrected']['test'])[:,1]
    print('Brier score pada test:',round(brier_score_loss(ytest,proba),4))
    bin_edges=np.linspace(0,1,7)
    ids=np.minimum(np.digitize(proba,bin_edges[1:-1]),5)
    rows=[]
    for k in range(6):
        mask=ids==k
        if mask.sum():rows.append((int(mask.sum()),float(proba[mask].mean()),float(ytest[mask].mean())))
    counts,conf,observed=map(np.asarray,zip(*rows))
    fig,ax=plt.subplots(figsize=(5.5,4))
    ax.plot([0,1],[0,1],'--',color='black',label='Kalibrasi ideal')
    ax.scatter(conf,observed,s=counts*7,alpha=.75,label='Bin terisi')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Rata-rata probabilitas prediksi',
           ylabel='Proporsi cacat teramati',title='Reliability diagram: TEST')
    ax.legend();plt.tight_layout();plt.show()
else:
    print('Model terpilih tidak menyediakan predict_proba; analisis kalibrasi dilewati.')

## 16. Audit biaya komputasi dan sensitivitas keputusan

Pilihan algoritma pada sistem instrumentasi juga bergantung pada latensi, memori, frekuensi akuisisi, dan biaya kalibrasi. Angka waktu dari notebook ini hanya estimasi lingkungan yang digunakan; pengukuran produksi memerlukan *warm-up*, profiling berulang, serta spesifikasi perangkat nyata.

Analisis sensitivitas *threshold* dilakukan hanya pada **validation**, bukan test, untuk menunjukkan bahwa satu titik operasi mungkin tidak cocok bagi biaya *false negative* tertentu. Pemilihan ambang klasifikasi selain 0,5 wajib dinyatakan sebelum evaluasi test.

In [ ]:
t0=time.perf_counter()
for _ in range(50):
    _=selected_model.predict(feature_banks['corrected']['validation'])
elapsed=(time.perf_counter()-t0)/50
print('Waktu prediksi rata-rata per batch validation:',round(elapsed*1000,3),'ms (lingkungan ini)')
if hasattr(selected_model,'predict_proba'):
    pv=selected_model.predict_proba(feature_banks['corrected']['validation'])[:,1]
    threshold_rows=[{'threshold':float(t),'F1_cacat_validasi':f1_score(yval,pv>=t,zero_division=0),
          'recall_cacat_validasi':recall_score(yval,pv>=t,zero_division=0)}
          for t in np.linspace(.1,.9,9)]
    print(pd.DataFrame(threshold_rows).round(3).to_string(index=False))

## 17. Integritas hasil: tes yang membuktikan kontrak eksperimen

Unit test berikut tidak membuktikan keselamatan atau kelayakan industri. Namun ia mendeteksi beberapa kegagalan mendasar yang sering terjadi: batch bercampur, label tidak seimbang tanpa sengaja, interpolasi buruk, kebocoran fitur ground truth, dan model yang melihat test selama tuning. Pernyataan terakhir hanya dapat diperiksa sebagian melalui audit kode; integritas proses juga memerlukan jejak keputusan peneliti.

In [ ]:
assert len(set(sum(BATCHES.values(),[])))==sum(map(len,BATCHES.values()))
assert all(len(d['labels'])==len(d['meta']) for d in datasets.values())
assert all(np.bincount(d['labels']).tolist()==[len(d['labels'])//2]*2 for d in datasets.values())
assert all(np.isfinite(d['images']).all() and np.isfinite(d['corrected']).all() for d in datasets.values())
assert dice(np.ones((4,4),bool),np.ones((4,4),bool))==1
assert abs(dice(np.zeros((4,4),bool),np.ones((4,4),bool)))<1e-9
assert all(len(FEATURE_NAMES)==feature_banks[k]['train'].shape[1] for k in feature_banks)
assert not any(k in FEATURE_NAMES for k in ['truth_mask','ground_truth','label','batch_id'])
assert set(np.unique(y_pred)).issubset({0,1})
assert selected_name in candidates
print('9 pemeriksaan integritas utama: LULUS')

## 18. Reproducibility dan provenance: membuat hasil bisa diperiksa

Dataset dari simulator wajib didampingi *dataset card*, kode generator, seed, definisi label, mask referensi, split yang terkunci, informasi satuan, dan penjelasan keterbatasan. File keluaran `.npz` berisi array; manifest `.json` merinci sumber dan batas pemakaian. Hash SHA-256 mendeteksi perubahan byte, tetapi **tidak** membuktikan mutu ilmiah, bebas bias, atau kebenaran anotasi.

Jika notebook dijalankan pada Colab, file akan ditulis di folder kerja runtime. Unduh berkas untuk menyimpannya atau jalankan ulang; runtime Colab tidak menjamin penyimpanan permanen. Ketika file sudah tersedia dalam repository, pengajar dapat mempertahankan generator sebagai sumber kebenaran dan memperlakukan NPZ sebagai artefak yang dihasilkan, bukan mengganti dataset tanpa melacak versinya.

In [ ]:
outdir=Path('output_notebook_12')
outdir.mkdir(exist_ok=True)
file_npz=outdir/'12_proyek_akhir_dataset_sintetis.npz'
arrays={}
for s,d in datasets.items():
    for k in ['images','labels','truth_masks','truth_refl','dark','flat']:
        arrays[f'{s}_{k}']=d[k]
np.savez_compressed(file_npz,**arrays)
sha=hashlib.sha256(file_npz.read_bytes()).hexdigest()
manifest={'judul':'Inspeksi kualitas tablet sintetis — Proyek Integratif 12',
    'generator':'Notebook 12, make_batch; citra sintetis, bukan data industri',
    'seed':SEED,'pixel_pitch_mm_asumsi':PITCH_MM,
    'pembagian_batch':BATCHES,'jumlah_citra':{s:len(d['labels']) for s,d in datasets.items()},
    'kelas':{'0':'tanpa cacat sintetis','1':'dengan cacat sintetis'},
    'sumber_referensi':'mask dan reflektansi dari simulator; hanya untuk evaluasi',
    'batas_klaim':'tidak tervalidasi pada kamera, tablet, atau protokol industri nyata',
    'sha256_npz':sha,'python':sys.version.split()[0],
    'numpy':np.__version__,'sklearn':sklearn.__version__}
file_manifest=outdir/'12_dataset_manifest.json'
file_manifest.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Dataset:',file_npz,'\nManifest:',file_manifest,'\nSHA-256:',sha[:24]+'...')

### 18.1 Memeriksa keterbacaan dan kesetaraan artefak

Membuat file tidak cukup: muat kembali dataset untuk memeriksa bentuk, kelas, dan konsistensi kontennya. Reproducibility juga mensyaratkan versi pustaka, benih acak, serta dokumentasi keputusan. Bila hasil berbeda setelah pembaruan pustaka, penyebabnya harus diaudit terlebih dahulu sebelum menyebut adanya kegagalan metode.

In [ ]:
with np.load(file_npz) as check:
    assert check['test_images'].shape==datasets['test']['images'].shape
    assert np.array_equal(check['test_labels'],datasets['test']['labels'])
    assert np.array_equal(check['train_truth_masks'],datasets['train']['truth_masks'])
    print('Artefak NPZ berhasil dibaca ulang:',len(check.files),'array')

## 19. Sintesis hasil dan batas kesimpulan

Diskusikan tiga pernyataan yang berbeda sebelum menyimpulkan keberhasilan:

1. **Kualitas citra:** apakah koreksi mengurangi galat radiometri terhadap referensi sintetis?
2. **Kualitas pengukuran:** apakah segmentasi mempertahankan luas dengan bias yang dapat diterima?
3. **Kualitas keputusan:** apakah fitur dan model mengurangi salah klasifikasi pada batch baru dan domain stress?

Ketiganya tidak saling menggantikan. Grafik yang tampak bersih tidak dapat menggantikan uji metrik. F1 tinggi tidak mengonfirmasi akurasi metrologi. Interval kepercayaan bootstrap tidak menghapus kesalahan sistematis dari simulator. Demikian pula koreksi dark–flat tidak otomatis mengatasi domain shift pada blur, kamera nonlinier, atau komposisi material.

**Batas klaim:** hasil notebook hanya berlaku pada generator, parameter, pembagian data, dan implementasi yang dijalankan. Sebelum aplikasi nyata, diperlukan protokol akuisisi, kalibrasi geometrik dan radiometrik, anotasi beberapa penilai, studi antarsesi/perangkat, audit label, ukuran sampel yang beralasan, serta validasi prospektif.

## 20. Latihan bertingkat dan tugas HOTS

**Latihan konseptual**

1. Jelaskan pada kondisi apa *flat field* dapat memperbesar noise, khususnya ketika $F-D$ sangat kecil.
2. Mengapa mengukur F1 piksel tidak setara dengan mengukur kesalahan diameter tablet?
3. Apa bedanya pengulangan delapan citra dalam satu batch dengan delapan batch independen?
4. Mengapa model klasik dapat lebih baik daripada CNN pada data terbatas tanpa menunjukkan bahwa CNN selalu buruk?

**Latihan analitis dan komputasional**

5. Turunkan propagasi ketidakpastian relatif $A=Np_xp_y$ untuk $p_x,p_y$ independen, dan diskusikan korelasi keduanya.
6. Bandingkan performa hasil fitur radiometri versus geometri saat noise sensor ditingkatkan dua kali. Lakukan tuning hanya melalui validation.
7. Simulasikan kasus satu batch mempunyai gain nonlinier $Y\propto R^{\gamma}$, lalu jelaskan kegagalan koreksi linear dark–flat.
8. Hitung bootstrap berbasis batch untuk bias luas, bukan hanya F1, dan interpretasikan perbedaan intervalnya.
9. Ubah prevalensi cacat pada test sintetis tanpa mengubah sensitivitas model: tunjukkan perubahan precision, lalu jelaskan implikasinya bagi inspeksi berbiaya tinggi.

**Tugas HOTS**

10. Rancang protokol akuisisi dan anotasi untuk 3 kamera dan 2 lokasi dengan pembagian data yang mencegah *leakage*. Tentukan unit independen dan kriteria sukses.
11. Buat argumen mengapa penggunaan mask ground truth sebagai ROI pelatihan merupakan kebocoran metodologis jika mask itu tidak tersedia pada deployment.
12. Susun matriks risiko sistem: false negative, error luas, waktu inferensi, kegagalan kalibrasi, serta dampak setiap risiko pada keputusan operasional.

**Produk laporan akhir:** artikel teknis ringkas dengan latar fisika, diagram pipeline, notasi dan asumsi, dataset card, pemisahan data, baseline, metrik segmentasi dan pengukuran, model terpilih, CI per batch, analisis stress, failure cases, keterbatasan, dan referensi.

## 21. Rubrik penilaian dan contoh struktur laporan

| Komponen | Bobot | Kriteria utama |
|---|---:|---|
| Model fisika dan asumsi | 15% | Persamaan konsisten, parameter dan domain validitas dinyatakan |
| Desain eksperimen dan data | 20% | Split independen, provenance, tidak ada leakage |
| Implementasi pipeline | 20% | Kode reproduktif, prapemrosesan dan evaluasi benar |
| Analisis metrologi | 15% | Dice/IoU, luas, bias, satuan dan ketidakpastian |
| Evaluasi ML dan generalisasi | 20% | Baseline, ablation, test/stress, ketidakpastian |
| Pelaporan dan integritas | 10% | Gambar terbaca, batas klaim jujur, sumber jelas |

**Struktur laporan 8–12 halaman yang disarankan:** abstrak; pendahuluan dan tujuan; model pembentukan citra; dataset dan protokol; metode; hasil; diskusi dan ancaman validitas; kesimpulan; referensi; lampiran kode atau lingkungan.

## 22. Referensi untuk pendalaman

1. Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, 4th ed., Pearson, 2018 — model, histogram, segmentasi, dan filtering.
2. Szeliski, R. *Computer Vision: Algorithms and Applications*, 2nd ed., Springer, 2022 — representasi, model citra, dan evaluasi sistem.
3. Bishop, C. M. *Pattern Recognition and Machine Learning*, Springer, 2006 — probabilitas, model dan regularisasi.
4. Hastie, T., Tibshirani, R., & Friedman, J. *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — model, validasi, dan generalisasi.
5. scikit-image documentation: https://scikit-image.org/docs/stable/ — implementasi citra dan pengukuran.
6. scikit-learn User Guide: https://scikit-learn.org/stable/user_guide.html — pipeline, metrik, validasi, dan model.
7. Peng, R. D. *Reproducible Research in Computational Science*, Science 334, 1226–1227 (2011), DOI: 10.1126/science.1213847.

Seluruh sumber dimaksudkan untuk penelusuran lebih lanjut; notebook tidak mengklaim memverifikasi semua rumus dan metrik terhadap eksperimen instrumen nyata.

## Koneksi berikutnya

**Pembentukan citra dan inverse problems** memperdalam ketakterbalikan blur serta noise; **optimisasi** menjelaskan fungsi objektif dan regularisasi; **Bayesian imaging** memperluas pengukuran ketidakpastian keputusan; **evaluasi ilmiah** membahas variasi anotasi, *domain shift*, ukuran efek, dan inferensi statistik; **reproducibility** menghubungkan seluruh proses ke audit provenance dan batas klaim. Karena itu proyek integratif ini bukan titik akhir, melainkan latihan untuk menyusun argumen ilmiah dari pengamatan hingga keputusan yang dapat diaudit.